# Solving a complex task with a multi-agent hierarchy

This notebook is part of the [Hugging Face Agents Course](https://www.hf.co/learn/agents-course), a free Course from beginner to expert, where you learn to build Agents.

![Agents course share](https://huggingface.co/datasets/agents-course/course-images/resolve/main/en/communication/share.png)

The reception is approaching! With your help, Alfred is now nearly finished with the preparations.

But now there's a problem: the Batmobile has disappeared. Alfred needs to find a replacement, and find it quickly.

Fortunately, a few biopics have been done on Bruce Wayne's life, so maybe Alfred could get a car left behind on one of the movie set, and re-engineer it up to modern standards, which certainly would include a full self-driving option.

But this could be anywhere in the filming locations around the world - which could be numerous.

So Alfred wants your help. Could you build an agent able to solve this task?

> 👉 Find all Batman filming locations in the world, calculate the time to transfer via a cargo plane to there, and represent them on a map, with a color varying by a cargo plane transfer time. Also represent some supercar factories with the same cargo plane transfer time.

Let's build this!

In [ ]:
!pip install 'smolagents[litellm]' plotly geopandas shapely kaleido -q

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
# We first make a tool to get the cargo plane transfer time.
import math
from typing import Optional, Tuple

from smolagents import tool


@tool
def calculate_cargo_travel_time(
    origin_coords: Tuple[float, float],
    destination_coords: Tuple[float, float],
    cruising_speed_kmh: Optional[float] = 750.0,  # Average speed for cargo planes
) -> float:
    """
    Calculate the travel time for a cargo plane between two points on Earth using great-circle distance.

    Args:
        origin_coords: Tuple of (latitude, longitude) for the starting point
        destination_coords: Tuple of (latitude, longitude) for the destination
        cruising_speed_kmh: Optional cruising speed in km/h (defaults to 750 km/h for typical cargo planes)

    Returns:
        float: The estimated travel time in hours

    Example:
        >>> # Chicago (41.8781° N, 87.6298° W) to Sydney (33.8688° S, 151.2093° E)
        >>> result = calculate_cargo_travel_time((41.8781, -87.6298), (-33.8688, 151.2093))
    """

    def to_radians(degrees: float) -> float:
        return degrees * (math.pi / 180)

    # Extract coordinates
    lat1, lon1 = map(to_radians, origin_coords)
    lat2, lon2 = map(to_radians, destination_coords)

    # Earth's radius in kilometers
    EARTH_RADIUS_KM = 6371.0

    # Calculate great-circle distance using the haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    )
    c = 2 * math.asin(math.sqrt(a))
    distance = EARTH_RADIUS_KM * c

    # Add 10% to account for non-direct routes and air traffic controls
    actual_distance = distance * 1.1

    # Calculate flight time
    # Add 1 hour for takeoff and landing procedures
    flight_time = (actual_distance / cruising_speed_kmh) + 1.0

    # Format the results
    return round(flight_time, 2)


print(calculate_cargo_travel_time((41.8781, -87.6298), (-33.8688, 151.2093)))

22.82


For the model provider, we use Together AI, one of the new [inference providers on the Hub](https://huggingface.co/blog/inference-providers)!

Regarding the GoogleSearchTool: this requires either having setup env variable `SERPAPI_API_KEY` and passing `provider="serpapi"` or having `SERPER_API_KEY` and passing `provider=serper`.

If you don't have any Serp API provider setup, you can use `WebSearchTool` but beware that it has a rate limit.

In [ ]:
import os
from PIL import Image
from smolagents import CodeAgent, GoogleSearchTool, InferenceClientModel, VisitWebpageTool

# model_id="Qwen/Qwen2.5-Coder-32B-Instruct"
# default is Qwen/Qwen3-Next-80B-A3B-Thinking
model = InferenceClientModel(model_id="Qwen/Qwen2.5-Coder-32B-Instruct")

We can start with creating a baseline, simple agent to give us a simple report.

In [ ]:
task = """Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're in Gotham, 40.7128° N, 74.0060° W), and return them to me as a pandas dataframe.
Also give me some supercar factories with the same cargo plane transfer time."""

In [ ]:
from google.colab import userdata
import os
os.environ["SERPAPI_API_KEY"] = userdata.get('SERPER_API_KEY')

In [ ]:
# Note: will see errors like "Model Qwen/Qwen2.5-Coder-32B-Instruct is not supported by provider fireworks-ai."
# https://discuss.huggingface.co/t/03-error-when-using-qwen-qwen2-5-vl-32b-instruct-with-inference-provider/168789/2
# seems to suggest it's an issue with smolAgent's InferenceClientModel
agent = CodeAgent(
    model=model,
    tools=[GoogleSearchTool(), VisitWebpageTool(), calculate_cargo_travel_time],
    additional_authorized_imports=["pandas"],
    max_steps=20,
)

In [ ]:
result = agent.run(task)

# A couple of interesting observations:
# (1) agent wrote its own calculate_travel_time function, it did not use thet tool we provided

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're   │
│ in Gotham, 40.7128° N, 74.0060° W), and return them to me as a pandas dataframe.                                │
│ Also give me some supercar factories with the same cargo plane transfer time.                                   │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  batman_locations = web_search("Batman filming locations worldwide")                                              
  print(batman_locations)                                                                                          
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
## Search Results
0. [The Batman | Film Locations](https://movie-locations.com/movies/b/The-Batman-2022-2.php)
Source: The Worldwide Guide To Movie Locations

The Batman | 2022 · Merseyside · London · Scotland · Bedfordshire · Hertfordshire · Essex · Surrey · Chicago ...

1. [Category:Film Locations | Batman Wiki - Fandom](https://batman.fandom.com/wiki/Category:Film_Locations)
Source: Batman Wiki

Category:Batman (1966 Movie) Locations · Category:Batman (1989 film) Locations · Category:Batman Forever Locations 
· Category:Batman Returns Locations.

2. [Where was The Batman filmed? ALL the Filming Locations 
...](https://www.atlasofwonders.com/2022/04/where-was-the-batman-filmed.html)
Source: Atlas of Wonders

Guide to the all the filming locations of The Batman - The 2022 movie with Robert Pattison filmed in Chicago and 
The UK - The City and the Wayne House.

3. [12 Batman Movie Locations You Can 
Visit!](https://www.travelandleisureasia.com/sea/destinations/batman-movie-locations-you-can-visit/)
Date published: Jan 10, 2023
Source: Travel and Leisure Asia

Most of the filming of Batman movies is done in the Warner Bros studios and across the US, including New York and 
Pittsburgh. You will notice ...

4. [The Batman (2022) - Filming & production](https://www.imdb.com/title/tt1877830/locations/)
Source: IMDb

Filming locations ; Necropolis Cemetery, Glasgow, Scotland, UK. (Batman and Selina leaving the cemetery) · 88 ; St.
George's Hall, Liverpool, England, UK. ( ...

5. [Batman Begins filming locations](https://moviemaps.org/movies/1r)
Source: MovieMaps

Batman Begins was filmed in London in the United Kingdom, Chicago in the United States of America, and in Iceland.

6. [What cities in America and anywhere in the world do you 
...](https://www.reddit.com/r/batman/comments/1d1t88q/what_cities_in_america_and_anywhere_in_the_world/)
Source: Reddit · r/batman

Glasgow and Liverpool were used in the shoot for The Batman. I'd avoid Chicago, as that looks more like Metropolis 
than Gotham.

7. [Batman Movies and Filming Locations in Los 
Angeles](https://www.traveltodayla.com/post/batman-movies-in-la?srsltid=AfmBOoqsbsGohmscMTAwrb4VOpO8w4VpS4O-39DE0SR
xFyEnzndU7GG6)
Date published: Apr 20, 2023
Source: LA Today

The legendary Batcave is found in Bronson Canyon, at the Bronson Caves. This is a section of the expansive Griffith
Park in Los Angeles that is ...

8. [Where Was The Dark Knight Rises Filmed? Complete 
...](https://giggster.com/guide/movie-location/where-was-the-dark-knight-rises-filmed)
Source: Giggster

The Dark Knight Rises was filmed in multiple locations around the world, with most scenes taking place in India, 
the United Kingdom, and the United States.

9. 
[iFILMthings](https://www.facebook.com/iFilmThings/posts/where-was-the-batman-filmed-the-batman-movie-was-filmed-in
-various-locations-aro/1472298814902230/)
Source: Facebook · iFILMthings

The Batman movie was filmed in various locations around the world, including London and Liverpool in England, 
Chicago in the United States, and ...

Out: None

[Step 1: Duration 8.41 seconds| Input tokens: 2,323 | Output tokens: 130]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 5 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Combine Batman filming locations and supercar factories within similar travel time range                       
  combined_data = pd.concat([batman_df, similar_factory_df], ignore_index=True)                                    
                                                                                                                   
  print(combined_data)                                                                                             
                                                                                                                   
  final_answer(combined_data)                                                                                      
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
                  Location               Coordinates  Travel Time (hours)
0               Merseyside  (53.4068294, -2.9671569)                 7.10
1                   London  (51.5073509, -0.1277583)                 7.43
2                 Scotland  (56.0759462, -3.8182046)                 6.94
3             Bedfordshire     (52.04022, -0.417923)                 7.38
4            Hertfordshire      (51.75194, -0.21767)                 7.41
5                    Essex      (51.70202, 0.114968)                 7.44
6                   Surrey  (51.2793065, -0.4837591)                 7.41
7                  Chicago       (41.8781, -87.6298)                 1.53
8         Morgan - Malvern   (52.243179, -2.1421187)                 7.22
9   Rolls-Royce - Goodwood  (51.0028683, -0.8972112)                 7.38
10   Aston Martin - Gaydon  (52.0767842, -0.8171987)                 7.34
11        McLaren - Woking    (51.360828, -0.549173)                 7.40

Final answer:                   Location               Coordinates  Travel Time (hours)
0               Merseyside  (53.4068294, -2.9671569)                 7.10
1                   London  (51.5073509, -0.1277583)                 7.43
2                 Scotland  (56.0759462, -3.8182046)                 6.94
3             Bedfordshire     (52.04022, -0.417923)                 7.38
4            Hertfordshire      (51.75194, -0.21767)                 7.41
5                    Essex      (51.70202, 0.114968)                 7.44
6                   Surrey  (51.2793065, -0.4837591)                 7.41
7                  Chicago       (41.8781, -87.6298)                 1.53
8         Morgan - Malvern   (52.243179, -2.1421187)                 7.22
9   Rolls-Royce - Goodwood  (51.0028683, -0.8972112)                 7.38
10   Aston Martin - Gaydon  (52.0767842, -0.8171987)                 7.34
11        McLaren - Woking    (51.360828, -0.549173)                 7.40

[Step 5: Duration 7.92 seconds| Input tokens: 23,808 | Output tokens: 1,732]

In [ ]:
result

,Location,Coordinates,Travel Time (hours)
0,Merseyside,"(53.4068294, -2.9671569)",7.10
1,London,"(51.5073509, -0.1277583)",7.43
2,Scotland,"(56.0759462, -3.8182046)",6.94
3,Bedfordshire,"(52.04022, -0.417923)",7.38
4,Hertfordshire,"(51.75194, -0.21767)",7.41
5,Essex,"(51.70202, 0.114968)",7.44
6,Surrey,"(51.2793065, -0.4837591)",7.41
7,Chicago,"(41.8781, -87.6298)",1.53
8,Morgan - Malvern,"(52.243179, -2.1421187)",7.22
9,Rolls-Royce - Goodwood,"(51.0028683, -0.8972112)",7.38


In [ ]:
!pip install markdownify requests
# install the dependency since model seems to be using these modules

We could already improve this a bit by throwing in some dedicated planning steps, and adding more prompting.

In [ ]:

# agent runs planning prompt every 4 steps of actions
agent.planning_interval = 4

detailed_report = agent.run(f"""
You're an expert analyst. You make comprehensive reports after visiting many websites.
Don't hesitate to search for many queries at once in a for loop.
For each data point that you find, visit the source url to confirm numbers.

{task}
""")

print(detailed_report)
# observations:
# (1) comparing to the previous prompt, this agent actually
# lists out the plan ahead of time and states that it will use the tool we provided. Similar to claude code!
# (2) Bro is taking his time lol. I'm killing the process after 12 minutes :)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You're an expert analyst. You make comprehensive reports after visiting many websites.                          │
│ Don't hesitate to search for many queries at once in a for loop.                                                │
│ For each data point that you find, visit the source url to confirm numbers.                                     │
│                                                                                                                 │
│ Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're   │
│ in Gotham, 40.7128° N, 74.0060° W), and return them to me as a pandas dataframe.                                │
│ Also give me some supercar factories with the same cargo plane transfer time.                                   │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

────────────────────────────────────────────────── Initial plan ───────────────────────────────────────────────────
Here are the facts I know and the plan of action that I will follow to solve the task:
```
## 1. Facts survey

### 1.1. Facts given in the task
- We are in Gotham, with coordinates: 40.7128° N, 74.0060° W

### 1.2. Facts to look up
- List of all Batman filming locations worldwide
  - Source: Likely multiple movie databases and fan wikis such as IMDb (https://www.imdb.com/) and Batman Wiki 
(https://batman.fandom.com/wiki/Batman_(film))
- Coordinates of each Batman filming location
  - Sources: Geographic data websites such as Google Maps (https://www.google.com/maps/) or official production 
company websites
- Typical cruising speed of a cargo plane
  - Source: Aviation data resources such as PilotFriend 
(https://aviationweather.gov/pilotfriend/data/briefings/cargo_speeds) or industry reports
- List of supercar factories worldwide
  - Sources: Car manufacturer official websites and automotive news sites like Forbes Automotive or Supercar 
Database
- Coordinates of each supercar factory
  - Sources: Same as above for coordinates of Batman filming locations

### 1.3. Facts to derive
- Travel times for transferring each Batman filming location to Gotham
  - Derivation: By using the `calculate_cargo_travel_time` function between the coordinates of Gotham and each 
filming location
- Grouping of supercar factories based on having similar travel times to the Batman filming locations
  - Derivation: Calculate travel times for each factory to Gotham and compare these times to those of the Batman 
filming locations to identify matches

## 2. Plan

1. Perform web searches to gather a list of all Batman filming locations worldwide.
2. Visit each source webpage of the Batman filming locations to extract and confirm their coordinates.
3. Web search to find the typical cruising speed of a cargo plane.
4. Confirm the cruising speed by visiting the source webpage and extracting the value.
5. Calculate the travel time for each Batman filming location to Gotham using the `calculate_cargo_travel_time` 
function.
6. Perform web searches to gather a list of supercar factories worldwide.
7. Visit each source webpage of the supercar factories to extract and confirm their coordinates.
8. Calculate the travel time for each supercar factory to Gotham using the `calculate_cargo_travel_time` function.
9. Create a pandas DataFrame containing Batman filming locations along with their travel times to Gotham.
10. Identify supercar factories whose travel times to Gotham match those of the Batman filming locations.
11. Add identified supercar factories to the existing pandas DataFrame.
12. 
```

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  batman_locations = web_search(query="batman filming locations")                                                  
  locations_list = batman_locations.split('\n')  # Split locations separated by newlines                           
                                                                                                                   
  # Extract unique, non-empty film locations from the search result                                                
  batman_filming_locations = list(set([loc.strip() for loc in locations_list if 'filming location' in loc.lower()  
  and loc.strip()]))                                                                                               
                                                                                                                   
  # Extract unique URLs for each film location and confirm coordinates                                             
  locations_urls = set()                                                                                           
  coordinates_dict Bat = {}                                                                                        
                                                                                                                   
  for location in batman_filming_locations:                                                                        
      search_result = web_search(query=location)                                                                   
      url = search_result.split('\n')[0].strip()  # Take the first URL in the result                               
                                                                                                                   
      if url and not url.startswith('http'):                                                                       
          continue  # Skip invalid URLs                                                                            
                                                                                                                   
      if url in locations_urls:                                                                                    
          continue  # Avoid duplicate URLs                                                                         
                                                                                                                   
      locations_urls.add(url)                                                                                      
                                                                                                                   
      # Visit the webpage and obtain geographic data                                                               
      page_content = visit_webpage(url=url)                                                                        
      lat_lon_pattern = r'Latitude:\s*([-+]?\d*\.\d+)\s*Longitude:\s*([-+]?\d*\.\d+)'                              
      lat_lon_match = re.search(pattern=lat_lon_pattern, string=page_content)                                      
                                                                                                                   
      if lat_lon_match:                                                                                            
          lat, lon = map(float, lat_lon_match.groups())                                                            
          coordinates_dict[location] = (lat, lon)                                                                  
                                                                                                                   
  print(coordinates_dict)                                                                                          
 ────────────────────────────────────────────────────────

Execution logs:
                  Location               Coordinates  Travel Time (hours)
0               Merseyside  (53.4068294, -2.9671569)                 7.10
1                   London  (51.5073509, -0.1277583)                 7.43
2                 Scotland  (56.0759462, -3.8182046)                 6.94
3             Bedfordshire     (52.04022, -0.417923)                 7.38
4            Hertfordshire      (51.75194, -0.21767)                 7.41
5                    Essex      (51.70202, 0.114968)                 7.44
6                   Surrey  (51.2793065, -0.4837591)                 7.41
7                  Chicago       (41.8781, -87.6298)                 1.53
8         Morgan - Malvern   (52.243179, -2.1421187)                 7.22
9   Rolls-Royce - Goodwood  (51.0028683, -0.8972112)                 7.38
10   Aston Martin - Gaydon  (52.0767842, -0.8171987)                 7.34
11        McLaren - Woking    (51.360828, -0.549173)                 7.40

...

────────────────────────────────────────────────── Updated plan ───────────────────────────────────────────────────
I still need to solve the task I was given:
```

You're an expert analyst. You make comprehensive reports after visiting many websites.
Don't hesitate to search for many queries at once in a for loop.
For each data point that you find, visit the source url to confirm numbers.

Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're in 
Gotham, 40.7128° N, 74.0060° W), and return them to me as a pandas dataframe.
Also give me some supercar factories with the same cargo plane transfer time.

```

Here are the facts I know and my new/updated plan of action to solve the task:
```
## 1. Updated facts survey
### 1.1. Facts given in the task
- We need to find all Batman filming locations worldwide.
- We must calculate the flight time from each filming location to Gotham City using a cargo plane.
- We need to provide the supercar factories with the closest transfer times as Gotham City.
- The output needs to be a pandas dataframe.

### 1.2. Facts that we have learned
- Extracting coordinates directly through a web search often yields incomplete or duplicated results leading to 
timeouts and errors in processing.
- Searching for specific film locations and factory coordinates on reliable sources like IMDb and Wikipedia 
improves accuracy.
- The function `calculate_cargo_travel_time` can be used to compute the travel time when provided with correct 
latitude and longitude.

### 1.3. Facts still to look up
- Confirmed coordinate sets for all Batman filming locations from reliable sources.
- Confirmed coordinate sets for each supercar factory.

### 1.4. Facts still to derive
- Flight times from each filming location and factory to Gotham City.
- A method to efficiently combine and filter Batman filming locations and supercar factories by similar transfer 
times.

## 2. Plan
### 2.1. Perform a web search for reliable sources of coordinates for all Batman filming locations.
### 2.2. Visit the pages from Step 2.1 and extract latitude and longitude for each filming location.
### 2.3. Perform a web search for reliable sources of coordinates for all considered supercar factories.
### 2.4. Visit the pages from Step 2.3 and extract latitude and longitude for each supercar factory.
### 2.5. Calculate flight times from each Batman filming location to Gotham City.
### 2.6. Calculate flight times from each supercar factory to Gotham City.
### 2.7. Combine calculated flight times from Steps 2.5 and 2.6 into a single dataframe.
### 2.8. Filter the combined dataframe to only include Batman filming locations and supercar factories with 
approximately the same flight time to Gotham City.

```

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 5 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  import re                                                                                                        
  import pandas as pd                                                                                              
                                                                                                                   
  # Define Gotham coordinates                                                                                      
  gotham_coords = (40.7128, -74.0060)                                                                              
                                                                                                                   
  # Batman films we are interested in                                                                              
  batman_films = [                                                                                                 
      "Batman Begins",                                                                                             
      "The Dark Knight",                                                                                           
      "The Dark Knight Rises",                                                                                     
      "Batman v Superman: Dawn of Justice",                                                                        
      "Suicide Squad",                                                                                             
      "Wonder Woman",                                                                                              
      "Justice League",                                                                                            
      "Zack Snyder's Justice League",                                                                              
      "The Lego Batman Movie",                                                                                     
      "The Batman (2022)"                                                                                          
  ]                                                                                                                
                                                                                                                   
  # Dictionary to store Batman filming locations and their coordinates                                             
  batman_locations_dict = {}                                                                                       
                                                                                                                   
  # Search and extract coordinates for each Batman film location                                                   
  for film in batman_films:                                                                                        
      filming_locations = web_search(query=f"{film} filming locations confirmed coordinates IMDb, Wikipedia")      
      locations_list = filming_locations.split('\n')                                                               
                                                                                                                   
      for loc in locations_list:                                                                                   
          loc = loc.strip()                                                                                        
          if not loc:                                                                                              
              continue                                                                                             
                                                                                                                   
          # Search IMDb and Wikipedia for more reliable r

[Step 5: Duration 259.19 seconds| Input tokens: 28,040 | Output tokens: 3,301]

KeyboardInterrupt: 

In [ ]:
detailed_report

,Location,Travel Time (hours)
0,"Bridge of Sighs, Glasgow Necropolis, Glasgow, ...",8.6
1,"Wishart Street, Glasgow, Scotland, UK",8.6


Thanks to these quick changes, we obtained a much more concise report by simply providing our agent a detailed prompt, and giving it planning capabilities!

💸 But as you can see, the context window is quickly filling up. So **if we ask our agent to combine the results of detailed search with another, it will be slower and quickly ramp up tokens and costs**.

➡️ We need to improve the structure of our system.

## ✌️ Splitting the task between two agents

Multi-agent structures allow to separate memories between different sub-tasks, with two great benefits:
- Each agent is more focused on its core task, thus more performant
- Separating memories reduces the count of input tokens at each step, thus reducing latency and cost.

Let's create a team with a dedicated web search agent, managed by another agent.

The manager agent should have plotting capabilities to redact its final report: so let us give it access to additional imports, including `plotly`, and `geopandas` + `shapely` for spatial plotting.

In [ ]:
model = InferenceClientModel(
    "Qwen/Qwen2.5-Coder-32B-Instruct", max_tokens=8096
)

os.environ["SERPER_API_KEY"] = userdata.get('SERPER_API_KEY')


web_agent = CodeAgent(
    model=model,
    tools=[
        GoogleSearchTool(provider="serper"),
        VisitWebpageTool(),
        calculate_cargo_travel_time,
    ],
    name="web_agent",
    description="Browses the web to find information",
    verbosity_level=0,
    max_steps=10,
)

The manager agent will need to do some mental heavy lifting.

So we give it the stronger model [DeepSeek-R1](https://huggingface.co/deepseek-ai/DeepSeek-R1), and add a `planning_interval` to the mix.

In [ ]:
# from google.colab import userdata
# import os
# ahhh don't wanna use OPENAI tokens
# os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [ ]:
from smolagents.utils import encode_image_base64, make_image_url
from smolagents import OpenAIServerModel


def check_reasoning_and_plot(final_answer, agent_memory):
    model = InferenceClientModel(model_id="deepseek-ai/DeepSeek-OCR-2", max_tokens=8096)
    # nah nah, we're not spending $$$ on gpt-4o
    # Here let's use any image-text to text model?? Not sure, let's try Kimi!!
    # multimodal_model = OpenAIServerModel("gpt-4o", max_tokens=8096)
    multimodal_model = model
    filepath = "saved_map.png"
    assert os.path.exists(filepath), "Make sure to save the plot under saved_map.png!"
    image = Image.open(filepath)
    prompt = (
        f"Here is a user-given task and the agent steps: {agent_memory.get_succinct_steps()}. Now here is the plot that was made."
        "Please check that the reasoning process and plot are correct: do they correctly answer the given task?"
        "First list reasons why yes/no, then write your final decision: PASS in caps lock if it is satisfactory, FAIL if it is not."
        "Don't be harsh: if the plot mostly solves the task, it should pass."
        "To pass, a plot should be made using px.scatter_map and not any other method (scatter_map looks nicer)."
    )
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": prompt,
                },
                {
                    "type": "image_url",
                    "image_url": {"url": make_image_url(encode_image_base64(image))},
                },
            ],
        }
    ]
    output = multimodal_model(messages).content
    print("Feedback: ", output)
    if "FAIL" in output:
        raise Exception(output)
    return True


manager_agent = CodeAgent(
    model=InferenceClientModel("deepseek-ai/DeepSeek-R1", max_tokens=8096),
    tools=[calculate_cargo_travel_time],
    managed_agents=[web_agent],
    additional_authorized_imports=[
        "geopandas",
        "plotly",
        "shapely",
        "json",
        "pandas",
        "numpy",
    ],
    planning_interval=5,
    verbosity_level=2,
    final_answer_checks=[check_reasoning_and_plot],
    max_steps=15,
)

Let us inspect what this team looks like:

In [ ]:
manager_agent.visualize()

CodeAgent | deepseek-ai/DeepSeek-R1
├── ✅ Authorized imports: ['geopandas', 'plotly', 'shapely', 'json', 'pandas', 'numpy']
├── 🛠️ Tools:
│   ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
│   ┃ Name                        ┃ Description                           ┃ Arguments                             ┃
│   ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│   │ calculate_cargo_travel_time │ Calculate the travel time for a cargo │ origin_coords (`array`): Tuple of     │
│   │                             │ plane between two points on Earth     │ (latitude, longitude) for the         │
│   │                             │ using great-circle distance.          │ starting point                        │
│   │                             │                                       │ destination_coords (`array`): Tuple   │
│   │                             │                                       │ of (latitude, longitude) for the      │
│   │                             │                                       │ destination                           │
│   │                             │                                       │ cruising_speed_kmh (`number`):        │
│   │                             │                                       │ Optional cruising speed in km/h       │
│   │                             │                                       │ (defaults to 750 km/h for typical     │
│   │                             │                                       │ cargo planes)                         │
│   │ final_answer                │ Provides a final answer to the given  │ answer (`any`): The final answer to   │
│   │                             │ problem.                              │ the problem                           │
│   └─────────────────────────────┴───────────────────────────────────────┴───────────────────────────────────────┘
└── 🤖 Managed agents:
    └── web_agent | CodeAgent | Qwen/Qwen2.5-Coder-32B-Instruct
        ├── ✅ Authorized imports: []
        ├── 📝 Description: Browses the web to find information
        └── 🛠️ Tools:
            ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
            ┃ Name                        ┃ Description                       ┃ Arguments                         ┃
            ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
            │ web_search                  │ Performs a google web search for  │ query (`string`): The search      │
            │                             │ your query then returns a string  │ query to perform.                 │
            │                             │ of the top search results.        │ filter_year (`integer`):          │
            │                             │                                   │ Optionally restrict results to a  │
            │                             │                                   │ certain year                      │
            │ visit_webpage               │ Visits a webpage at the given url │ url (`string`): The url of the    │
            │                             │ and reads its content as a        │ webpage to visit.                 │
            │                             │ markdown string. Use this to      │                                   │
            │                             │ browse webpages.                  │                                   │
            │ calculate_cargo_travel_time │ Calculate the travel time for a   │ origin_coords (`array`): Tuple of │
            │                             │ cargo plane between two points on │ (latitude, longitude) for the     │
            │                             │ Earth using great-circle          │ starting point                    │
            │                             │ distance.     

In [ ]:
manager_agent.run("""
Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're in Gotham, 40.7128° N, 74.0060° W).
Also give me some supercar factories with the same cargo plane transfer time. You need at least 6 points in total.
Represent this as spatial map of the world, with the locations represented as scatter points with a color that depends on the travel time, and save it to saved_map.png!

Here's an example of how to plot and return a map:
import plotly.express as px
df = px.data.carshare()
fig = px.scatter_map(df, lat="centroid_lat", lon="centroid_lon", text="name", color="peak_hour", size=100,
     color_continuous_scale=px.colors.sequential.Magma, size_max=15, zoom=1)
fig.show()
fig.write_image("saved_image.png")
final_answer(fig)

Never try to process strings using code: when you have a string to read, just print it and you'll see it.
""")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're   │
│ in Gotham, 40.7128° N, 74.0060° W).                                                                             │
│ Also give me some supercar factories with the same cargo plane transfer time. You need at least 6 points in     │
│ total.                                                                                                          │
│ Represent this as spatial map of the world, with the locations represented as scatter points with a color that  │
│ depends on the travel time, and save it to saved_map.png!                                                       │
│                                                                                                                 │
│ Here's an example of how to plot and return a map:                                                              │
│ import plotly.express as px                                                                                     │
│ df = px.data.carshare()                                                                                         │
│ fig = px.scatter_map(df, lat="centroid_lat", lon="centroid_lon", text="name", color="peak_hour", size=100,      │
│      color_continuous_scale=px.colors.sequential.Magma, size_max=15, zoom=1)                                    │
│ fig.show()                                                                                                      │
│ fig.write_image("saved_image.png")                                                                              │
│ final_answer(fig)                                                                                               │
│                                                                                                                 │
│ Never try to process strings using code: when you have a string to read, just print it and you'll see it.       │
│                                                                                                                 │
╰─ InferenceClientModel - deepseek-ai/DeepSeek-R1 ────────────────────────────────────────────────────────────────╯

ReadTimeout: (ReadTimeoutError("HTTPSConnectionPool(host='router.huggingface.co', port=443): Read timed out. (read timeout=120)"), '(Request ID: e2316a06-fa75-43a1-8583-5f302f3c2031)')

I don't know how that went in your run, but in mine, the manager agent skilfully divided tasks given to the web agent in `1. Search for Batman filming locations`, then `2. Find supercar factories`, before aggregating the lists and plotting the map.

Let's see what the map looks like by inspecting it directly from the agent state:

In [ ]:
manager_agent.python_executor.state["fig"]

![output map](https://huggingface.co/datasets/agents-course/course-images/resolve/main/en/unit2/smolagents/output_map.png)